In [1]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA: True
GPU: Tesla T4


In [3]:
import glob
import zipfile
import os

zip_files = glob.glob("/kaggle/input/**/rdd2022_yolo.zip", recursive=True)

print("Found:", zip_files)

zip_path = zip_files[0]
extract_path = "/kaggle/working"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print(os.listdir("/kaggle/working/rdd2022_yolo"))

Found: ['/kaggle/input/notebooks/adnanbaothman/notebook225b511d78/rdd2022_yolo.zip']
['val', 'train', 'test', 'data.yaml']


In [4]:
yaml_path = "/kaggle/working/rdd2022_yolo/data.yaml"

with open(yaml_path, "w") as f:
    f.write("""path: /kaggle/working/rdd2022_yolo
train: train/images
val: val/images
test: test/images

names:
  0: D00
  1: D10
  2: D20
  3: D40
""")

print(open(yaml_path).read())

path: /kaggle/working/rdd2022_yolo
train: train/images
val: val/images
test: test/images

names:
  0: D00
  1: D10
  2: D20
  3: D40



In [5]:
try:
    import gdown
except ImportError:
    !pip install -q gdown
    import gdown

url = "https://drive.google.com/file/d/1PUJyCImVsZTiv4DLFGOydjnyYjsLrK3D/view?usp=sharing"

gdown.download(
    url,
    "/kaggle/working/last.pt",
    quiet=False,
    fuzzy=True
)

print("Downloaded:", os.path.exists("/kaggle/working/last.pt"))

Downloading...
From (original): https://drive.google.com/uc?id=1PUJyCImVsZTiv4DLFGOydjnyYjsLrK3D
From (redirected): https://drive.google.com/uc?id=1PUJyCImVsZTiv4DLFGOydjnyYjsLrK3D&confirm=t&uuid=e2a5393a-faf5-48d5-8d70-b8bc21f5c754
To: /kaggle/working/last.pt
100%|██████████| 57.0M/57.0M [00:00<00:00, 57.4MB/s]

Downloaded: True


In [6]:
!pip install -q ultralytics==8.4.135

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.7 MB/s eta 0:00:00


In [7]:
import os

os.makedirs("/kaggle/working/runs", exist_ok=True)
os.makedirs("/content/drive/MyDrive/NEXORA/RoadDamageProject", exist_ok=True)

old_path = "/content/drive/MyDrive/NEXORA/RoadDamageProject/runs"

if os.path.lexists(old_path):
    os.remove(old_path) if os.path.islink(old_path) else None

if not os.path.exists(old_path):
    os.symlink("/kaggle/working/runs", old_path)

print("Save path ready")

Save path ready


In [8]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/last.pt")

print("Saved epoch:", model.ckpt["epoch"])

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Saved epoch: 18


In [9]:
results = model.train(
    resume=True,
    data="/kaggle/working/rdd2022_yolo/data.yaml",
    device=0
)

Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rdd2022_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_ba

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      21/50      7.89G      1.712      1.788      1.508         46        640: 100% ━━━━━━━━━━━━ 960/960 1.7it/s 9:360.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 1.8it/s 33.5s0.6ss
                   all       3833       5445      0.594      0.519      0.548      0.278

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      22/50       7.9G       1.71       1.78      1.509         54        640: 100% ━━━━━━━━━━━━ 960/960 1.7it/s 9:360.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 1.8it/s 34.2s0.6ss
                   all       3833       5445       0.59      0.522      0.556      0.284

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      23/50       7.9G      1.702      1.762        1.5         37        640: 100% ━━━━━━━━━━━━ 960/960 1.7it/s 9:360.5ss
                 Class     Im

In [10]:
import shutil

shutil.make_archive(
    "/kaggle/working/yolo11s_baseline_complete",
    "zip",
    "/kaggle/working/runs/yolo11s_baseline"
)

print("Backup created")

Backup created
